In [49]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------- -------------------------------- 2.4/12.8 MB 24.2 MB/s eta 0:00:01
     ----------------------------------- --- 11.8/12.8 MB 38.1 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 36.1 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [50]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
import numpy as np
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import wordcloud

In [35]:
# modeli za ustrezno obdelavo stavkov, besed, ločil....
nltk.download('punkt')     # stavki, besede
nltk.download('wordnet') #lemmatizacija
nltk.download('averaged_perceptron_tagger') #POS tagganje
nltk.download('omw-1.4') 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [36]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [37]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [38]:
# tokenization and lemmatization 
lemmatizer= WordNetLemmatizer()

In [39]:
# pokupčkamo besede s podobnim korenom, pomenom skupaj
# run, runs, running -> run
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('RB'):
        return wordnet.ADV
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    else:
        return wordnet.NOUN

In [56]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [57]:
def tokenize_lematize(tekst):
    doc = nlp(tekst)
    
    izbrane_besede = []
    
    # želimo odstraniti (osebe, kraji, jeziki, narodi)
    odstrani_entitete = {'PERSON', 'GPE', 'LOC', 'NORP', 'FAC', 'ORG'}
    
    # ustvarimo množico besed, ki so del prepoznanih entitet (imen)
    imena_v_tekstu = {ent.text.lower() for ent in doc.ents if ent.label_ in odstrani_entitete}

    for token in doc:
        # če je beseda ime (NER)
        if token.text.lower() in imena_v_tekstu:
            continue
            
        # spaCy uporablja oznake: NOUN ADJ
        if token.pos_ in ['NOUN', 'ADJ']:
            beseda = token.lemma_.lower()
            
            # samo črke in dolžina nad 2 znaka
            if beseda.isalpha() and len(beseda) > 2:
                izbrane_besede.append(beseda)
                
    return izbrane_besede

In [52]:
import random
from sklearn.feature_extraction import text

In [145]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()


custom_words = {
    # založniški šum 
    'book', 'novel', 'story', 'author', 'literature', 'edition', 'seller', 
    'read', 'reader', 'page', 'chapter', 'write', 'writer', 'publish', 
    'publication', 'review', 'times', 'york', 'print', 'copy', 'best', 
    'original', 'classic', 'introduction', 'note', 'debut', 'thriller',
    'unique', 'fascinating', 'scandal', 'major',
    
    # splošni opisi 
    'way', 'thing', 'important', 'practical', 'young', 'boy', 'girl', 
    'human', 'people', 'great', 'good', 'bad', 'true', 'new', 'old',
    'life', 'world', 'everything', 'day', 'time', 'year', 'make',
    'take', 'come', 'think', 'feel', 'know', 'look', 'want', 'large', 'small',
    'man', 'woman', 'literary', 'secret', 
    
    #iz izpisa
    'professional', 'guide', 'experience', 'natural', 'vivid', 'narrative',
    'compelling', 'extraordinary', 'powerful', 'voice', 'mind'
}


all_stopwords = list(base_stopwords.union(custom_words))

In [164]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)

#random.seed(42)
# vzamemo 150/200 knjig, ostale bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\leto_2003\03_ang_opisi\*.txt')[:150]
#filepaths= random.sample(filepaths, 150)
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
# custom_stopwords = list(text.ENGLISH_STOP_WORDS.union({'book', 'novel', 'story', 'author', 'character',
#                                                   'edition', 'classic', 'bestseller', 'review',
#                                                   'read', 'reader', 'york', 'times', 'new'}))
vectorizer= CountVectorizer(stop_words= all_stopwords, #custom_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=3, 
                            max_df=0.7)

In [165]:
X = vectorizer.fit_transform(filepaths) 


c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [166]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2977 stored elements and shape (150, 569)>
  Coords	Values
  (0, 171)	1
  (0, 132)	2
  (0, 415)	1
  (0, 343)	1
  (0, 359)	1
  (0, 424)	1
  (0, 379)	1
  (0, 527)	1
  (0, 364)	1
  (0, 388)	1
  (0, 219)	1
  (0, 196)	2
  (0, 482)	1
  (0, 106)	1
  (0, 480)	1
  (0, 125)	1
  (0, 206)	1
  (0, 1)	1
  (0, 114)	1
  (0, 505)	1
  (0, 124)	1
  (0, 377)	1
  (0, 301)	1
  (1, 132)	1
  (1, 206)	1
  :	:
  (147, 552)	3
  (147, 213)	1
  (147, 123)	1
  (147, 88)	1
  (147, 67)	3
  (147, 529)	1
  (147, 248)	1
  (147, 344)	1
  (147, 54)	1
  (147, 284)	1
  (147, 339)	1
  (147, 458)	1
  (148, 343)	1
  (148, 359)	1
  (148, 385)	1
  (148, 73)	1
  (149, 364)	1
  (149, 106)	1
  (149, 230)	1
  (149, 335)	1
  (149, 432)	1
  (149, 119)	1
  (149, 568)	2
  (149, 57)	1
  (149, 182)	2
[[0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 2]]
   account  action  actual  addition  adolescence  adventure  

In [149]:
def nmf(X, k, max_iter=500, tol=1e-4, random_state=42):
    """
    Nenegativna matrična faktorizacija, ki uporablja pravila za posodobitev elementov na podlagi množenja.

    Parametri:
    -----------
    X : ndarray (m x n)
        Nenegativna matrika
    k : int
        Stevilo komponent (teme/ žanri)
    max_iter : int
        Maksimalno število iteracij
    tol : float
        Toleranca konvergence (izračunano s pomočjo reconstruction error)
    random_state : int
        

    Vrne:
    --------
    W : ndarray (m x k)
    H : ndarray (k x n)
    errors : list
        Reconstruction za vsako iteracijo
    """
    
    np.random.seed(random_state)
    
    m, n = X.shape
    
    #zacnemo z nakljucnimi nenegativnimi vrednostmi
    W = np.random.rand(m, k)
    H = np.random.rand(k, n)
    
    eps = 1e-9
    errors = []
    
    for i in range(max_iter):
        
        # posodabljanje H
        H *= (W.T @ X) / (W.T @ W @ H + eps) # + eps, da ne delimo z 0
        # posodabljanje W
        W *= (X @ H.T) / (W @ (H @ H.T) + eps)
        
        # reconstruction error
        error = np.linalg.norm(X - W @ H, 'fro')
        errors.append(error)
        
        # konvergenca
        if i > 0 and abs(errors[-2] - error) < tol:
            break

    return W, H, errors

In [167]:
# test 

W, H, errors = nmf(X, 4)
print(errors)
#print(W)
#print(H)

[np.float64(67.01788502401399), np.float64(66.48690801392351), np.float64(66.06193028244789), np.float64(65.6014561929372), np.float64(65.18473829622152), np.float64(64.89388234875125), np.float64(64.7150639159821), np.float64(64.59995203177195), np.float64(64.52011354025117), np.float64(64.4606223475653), np.float64(64.41377913496144), np.float64(64.3758536889882), np.float64(64.34426287563242), np.float64(64.31719380928901), np.float64(64.29412556699947), np.float64(64.27389511764015), np.float64(64.25579135085191), np.float64(64.23924435917422), np.float64(64.22399683380618), np.float64(64.20995658350823), np.float64(64.19722108768936), np.float64(64.18553596500028), np.float64(64.17438249467982), np.float64(64.16352238371405), np.float64(64.15286060122295), np.float64(64.14230809005795), np.float64(64.1319692319238), np.float64(64.12194474754642), np.float64(64.11226232036692), np.float64(64.10288470089921), np.float64(64.09372208479614), np.float64(64.08484015831753), np.float64(6

In [168]:
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[-15:]]
    print(f"Tema {topic_idx +1}: {' '.join(top_words)}")

Tema 1: account authority realm belief leader savage violence church work bone religious brother violent fundamentalist faith
Tema 2: social remarkable marriage power city place friend school medical prison heart child mother family love
Tema 3: ambition work bloody different resource era epic revolution place empire country camp historical war history
Tema 4: power job dead dream century enemy business creative truth death key vampire night character dark


In [159]:
import os

# iz W dobim, kateri temi pripada katera knjiga, dobimo indeks najvišje vrednosti v vsaki vrstici
dominant_topics = np.argmax(W, axis=1)+1

filenames = [os.path.basename(f) for f in filepaths]

#tabela
df_results = pd.DataFrame({
    'Knjiga': filenames,
    'Tema': dominant_topics
})
print(df_results.head(10))
print(df_results[df_results['Tema'] == 4])
print(df_results[df_results['Tema'] == 5]) # za specif. teme
print(df_results[df_results['Tema'] == 2])


         Knjiga  Tema
0    opis_1.txt     4
1   opis_10.txt     2
2  opis_100.txt     4
3  opis_101.txt     4
4  opis_102.txt     2
5  opis_103.txt     4
6  opis_104.txt     4
7  opis_105.txt     4
8  opis_106.txt     3
9  opis_107.txt     4
           Knjiga  Tema
0      opis_1.txt     4
2    opis_100.txt     4
3    opis_101.txt     4
5    opis_103.txt     4
6    opis_104.txt     4
..            ...   ...
145   opis_59.txt     4
146    opis_6.txt     4
147   opis_60.txt     4
148   opis_61.txt     4
149   opis_62.txt     4

[82 rows x 2 columns]
Empty DataFrame
Columns: [Knjiga, Tema]
Index: []
           Knjiga  Tema
1     opis_10.txt     2
4    opis_102.txt     2
13   opis_110.txt     2
17   opis_114.txt     2
20   opis_117.txt     2
24   opis_120.txt     2
39   opis_134.txt     2
43   opis_138.txt     2
44   opis_139.txt     2
48   opis_142.txt     2
52   opis_146.txt     2
53   opis_147.txt     2
55   opis_149.txt     2
56    opis_15.txt     2
59   opis_152.txt     2
63   opis_156